# SDK-Sovereign Teacher Distillation Notebook

**Goal:** Produce a real, working trained adapter in under 90 minutes on a Colab T4.

## Why this notebook exists

Your previous training runs got 0% pass rate because:
- RL needs reward variance (most of yours was negative)
- 12-40 step training is too short for RL to find good policy
- The model never saw a clean example of "correct patch + correct verification flow"

This notebook takes a different path: **teacher distillation via SFT.**

1. The rule/teacher policy in your codebase already knows the correct moves
2. We generate clean demonstrations from it across all repos
3. We SFT both adapters on those demonstrations (not RL)
4. We eval the result

SFT on expert data is reliable. RL on noisy rewards is not. For a hackathon timeline, SFT wins every time.

## What gets saved permanently

- `huggingface.co/<HF_USER>/sdk-sovereign-distilled-lead` — Lead adapter
- `huggingface.co/<HF_USER>/sdk-sovereign-distilled-auditor` — Auditor adapter
- `huggingface.co/<HF_USER>/sdk-sovereign-checkpoints` — eval results, plots, logs

## Time budget on T4

| Cell | What | Time |
|---|---|---|
| 1-3 | Setup, env, model | ~5 min |
| 4 | Generate teacher demos | ~3 min |
| 5 | Baseline eval | ~5 min |
| 6 | SFT Lead adapter | ~15 min |
| 7 | SFT Auditor adapter | ~10 min |
| 8 | Trained eval | ~5 min |
| 9 | Plots + bundle | ~3 min |
| **Total** | | **~50 min** |

## Required Colab Secrets
- `HF_TOKEN` (write-scoped)
- `WANDB_API_KEY` (optional)


## Cell 1 — Install dependencies

In [ ]:
# Cell 1 — pinned stack
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U "trl==0.24.0" "datasets==3.6.0" "accelerate==1.6.0" "pydantic==2.10.6" peft bitsandbytes
!pip install -q "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv@v0.2.3"
!pip install -q wandb huggingface_hub matplotlib numpy
print('OK: dependencies installed')

## Cell 2 — Setup, secrets, repo clone, env server

In [ ]:
# Cell 2 — config + auth + clone + start localhost env
import os
import sys
import json
import subprocess
import time
import warnings
import requests
from pathlib import Path
from datetime import datetime

from google.colab import userdata
from huggingface_hub import login, HfApi, create_repo

warnings.filterwarnings('ignore')

# === EDIT IF NEEDED ===
GH_REPO = 'ishansurdi/SDK-Sovereign'
HF_USER = 'ishansurdi'
WANDB_PROJECT = 'sdk-sovereign-distilled'

# === Derived paths ===
LEAD_REPO = f'{HF_USER}/sdk-sovereign-distilled-lead'
AUDITOR_REPO = f'{HF_USER}/sdk-sovereign-distilled-auditor'
ARTIFACTS_REPO = f'{HF_USER}/sdk-sovereign-distilled-checkpoints'

# === Eval scale (keep small but big enough for stable pass rate) ===
N_EVAL_EPISODES_PER_REPO = 6  # 3 repos x 6 = 18 episodes per phase
N_TEACHER_EPISODES_PER_REPO = 12  # 36 total demonstrations

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'RUN_ID: {RUN_ID}')

# === Local paths ===
BASE = Path('/content')
LOGS = BASE / 'logs'
ARTIFACTS = BASE / 'artifacts'
PLOTS = BASE / 'plots'
for d in [LOGS, ARTIFACTS, PLOTS]:
    d.mkdir(parents=True, exist_ok=True)

# === Auth ===
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
os.environ['HF_TOKEN'] = hf_token

try:
    wandb_key = userdata.get('WANDB_API_KEY')
    if wandb_key:
        os.environ['WANDB_API_KEY'] = wandb_key
        import wandb
        wandb.login(key=wandb_key)
        USE_WANDB = True
        print('OK: W&B authenticated')
    else:
        USE_WANDB = False
except Exception:
    USE_WANDB = False
    print('No W&B key, continuing without it')

# === Clone ===
if not Path('sdk_sovereign_pkg').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{GH_REPO}.git', 'sdk_sovereign_pkg'], check=True)
else:
    subprocess.run(['git', 'pull'], cwd='sdk_sovereign_pkg', check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'sdk_sovereign_pkg/'], check=True)
sys.path.insert(0, 'sdk_sovereign_pkg')

# === Create HF Hub repos ===
api = HfApi()
for repo in [LEAD_REPO, AUDITOR_REPO, ARTIFACTS_REPO]:
    try:
        create_repo(repo, repo_type='model', private=False, exist_ok=True)
        print(f'  HF repo ready: {repo}')
    except Exception as e:
        print(f'  WARN {repo}: {e}')

# === Start localhost env server ===
subprocess.run(['pkill', '-f', 'uvicorn'], check=False)
time.sleep(2)

candidate_modules = [
    'server.app:app',
    'sdk_sovereign.server.app:app',
    'openenv_sdk_sovereign.server.app:app',
]
env_proc = None
ENV_URL = None
for module_path in candidate_modules:
    try:
        proc = subprocess.Popen(
            ['uvicorn', module_path, '--host', '0.0.0.0', '--port', '8000'],
            cwd='sdk_sovereign_pkg',
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        time.sleep(8)
        r = requests.get('http://localhost:8000/health', timeout=5)
        if r.status_code == 200:
            env_proc = proc
            ENV_URL = 'http://localhost:8000'
            print(f'OK: localhost env running, module={module_path}')
            break
        proc.terminate()
    except Exception:
        try:
            proc.terminate()
        except Exception:
            pass

if ENV_URL is None:
    ENV_URL = 'https://ishansurdi-sdk-sovereign.hf.space'
    print(f'WARN: localhost env failed to start, using remote {ENV_URL}')

print(f'\nENV_URL: {ENV_URL}')
print('OK: setup complete')

## Cell 3 — Load model and helpers

In [ ]:
# Cell 3 — load model + define helpers
import torch
from server.llm_agents import load_model_with_two_adapters, make_agent_pair
from client import SDKSovereignEnv
from models import SDKAction, SDKObservation

# Helper: push to Hub safely (never raises)
def push_to_hub_safe(folder_path, repo_id, commit_msg, path_in_repo=None):
    try:
        kw = {
            'folder_path': str(folder_path),
            'repo_id': repo_id,
            'repo_type': 'model',
            'commit_message': commit_msg,
        }
        if path_in_repo:
            kw['path_in_repo'] = path_in_repo
        api.upload_folder(**kw)
        print(f'  pushed to https://huggingface.co/{repo_id}')
        return True
    except Exception as e:
        print(f'  WARN push failed: {e}')
        return False


def save_jsonl(path, rows):
    Path(path).write_text('\n'.join(json.dumps(r) for r in rows))


def load_jsonl(path):
    p = Path(path)
    if not p.exists():
        return []
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()]


# === Load model ===
print('Loading model...')
model, tokenizer = load_model_with_two_adapters()
agents = make_agent_pair(model, tokenizer)
print(f'OK: model loaded, VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'Available adapters: {list(model.peft_config.keys())}')

## Cell 4 — Generate teacher demonstrations

This is the critical cell. We use the existing teacher/rule policy to generate
clean (observation, action) pairs across all repos. These become our SFT dataset.

The teacher policy is deterministic and ALREADY produces successful episodes -
we just need to capture its actions per role per scenario.

In [ ]:
# Cell 4 — generate teacher demonstrations
# Strategy: for each repo, run the teacher policy, capture every (obs, action)
# pair, format as instruction-tuning rows.

# Try to import teacher policy. The exact import depends on your codebase.
# We'll try a few common locations.

teacher_policy = None
try:
    from server.policy_runtime import get_policy
    teacher_policy = get_policy('teacher')
    print('Found teacher via policy_runtime')
except Exception as e:
    print(f'policy_runtime path failed: {e}')

if teacher_policy is None:
    try:
        from server.teacher import teacher_action
        teacher_policy = teacher_action
        print('Found teacher via server.teacher')
    except Exception as e:
        print(f'server.teacher path failed: {e}')

# Fallback: build a teacher inline using the deterministic rule for each repo
if teacher_policy is None:
    print('Falling back to inline teacher rules...')

    REPO_RULES = {
        'maps_repo': {'forbidden': 'googlemaps', 'replacement': 'mmi_sdk'},
        'payments_repo': {'forbidden': 'stripe', 'replacement': 'razorpay'},
        'comms_repo': {'forbidden': 'twilio', 'replacement': 'kaleyra'},
    }

    def teacher_policy(obs):
        """Hand-rolled deterministic policy that should win every episode."""
        if obs.current_role == 'auditor':
            # On turn 0: pass to let Lead see the code
            if obs.turn_index == 0:
                return SDKAction(role='auditor', action_type='pass')
            # If Lead just proposed: check the proposal against allow-list
            if obs.current_proposal is not None:
                allowed = obs.visible_allowlist or []
                proposed = getattr(obs.current_proposal, 'proposed_sdk',
                                   obs.current_proposal if isinstance(obs.current_proposal, str) else '')
                if proposed in allowed:
                    return SDKAction(role='auditor', action_type='approve')
                else:
                    return SDKAction(role='auditor', action_type='reject',
                                     rejection_reason=f'{proposed} not on sovereign allowlist')
            # Otherwise: pass
            return SDKAction(role='auditor', action_type='pass')

        elif obs.current_role == 'lead':
            # Detect which repo from the codebase
            code = (obs.visible_codebase or '').lower()
            for repo_id, rule in REPO_RULES.items():
                if rule['forbidden'].lower() in code:
                    forbidden = rule['forbidden']
                    replacement = rule['replacement']
                    break
            else:
                # Default: try razorpay
                forbidden = 'stripe'
                replacement = 'razorpay'

            # First turn for Lead: propose
            if obs.approved_replacement is None:
                return SDKAction(role='lead', action_type='propose_replacement',
                                 proposed_sdk=replacement)
            # Auditor approved: submit patch
            patched = (obs.visible_codebase or '').replace(forbidden, replacement)
            return SDKAction(role='lead', action_type='submit_patch',
                             patched_code=patched)

        return SDKAction(role=obs.current_role, action_type='pass')

    print('OK: inline teacher built')

# === Run teacher across repos ===
demos = []  # list of (role, prompt, completion_action_json) triples
repos_to_try = ['maps_repo', 'payments_repo', 'comms_repo']

print(f'\nRunning teacher across {len(repos_to_try)} repos x {N_TEACHER_EPISODES_PER_REPO} eps...')
total_success = 0

for repo_id in repos_to_try:
    print(f'\n--- {repo_id} ---')
    repo_success = 0
    for ep in range(N_TEACHER_EPISODES_PER_REPO):
        try:
            with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                # Some env APIs let you specify repo_id, others don't
                try:
                    obs = env.reset(repo_id=repo_id)
                except TypeError:
                    obs = env.reset()

                episode_demos = []
                while not obs.done and obs.turn_index < 7:
                    # Build a prompt for SFT
                    role_agent = agents[obs.current_role]
                    prompt = role_agent._build_prompt(obs)

                    # Get teacher action (the perfect one)
                    action = teacher_policy(obs)

                    # Format the action as the completion the model SHOULD produce
                    completion = json.dumps(action.model_dump() if hasattr(action, 'model_dump') else action.dict())

                    episode_demos.append({
                        'role': obs.current_role,
                        'prompt': prompt,
                        'completion': completion,
                        'action_type': action.action_type,
                        'turn': obs.turn_index,
                        'repo_id': repo_id,
                    })

                    obs = env.step(action)

                # Check if episode succeeded
                state = env.state()
                tr = state.test_results or {}
                success = bool(tr and all(tr.values()))
                if success:
                    repo_success += 1
                    total_success += 1
                    # ONLY add demonstrations from successful episodes
                    demos.extend(episode_demos)

        except Exception as e:
            print(f'  ep {ep} crashed: {e}')

    print(f'  {repo_id}: {repo_success}/{N_TEACHER_EPISODES_PER_REPO} successful')

print(f'\n=== Teacher generation summary ===')
print(f'Total successful episodes: {total_success}/{len(repos_to_try) * N_TEACHER_EPISODES_PER_REPO}')
print(f'Total (obs, action) demonstrations collected: {len(demos)}')

if total_success == 0:
    print('\nERROR: teacher policy produced 0 successful episodes.')
    print('The teacher_policy function above does not match your environment.')
    print('Check what your env expects for SDKAction fields and adjust the inline teacher.')
    raise RuntimeError('No successful demos to train on')

# === Split by role and save ===
lead_demos = [d for d in demos if d['role'] == 'lead']
auditor_demos = [d for d in demos if d['role'] == 'auditor']

print(f'\nLead demos: {len(lead_demos)}')
print(f'Auditor demos: {len(auditor_demos)}')

save_jsonl(LOGS / f'demos_lead_{RUN_ID}.jsonl', lead_demos)
save_jsonl(LOGS / f'demos_auditor_{RUN_ID}.jsonl', auditor_demos)

# Backup to Hub immediately - this is your most valuable artifact
push_to_hub_safe(LOGS, ARTIFACTS_REPO, f'teacher demos {RUN_ID}', path_in_repo=f'demos_{RUN_ID}')
print('\nOK: teacher demonstrations ready and backed up to Hub')

## Cell 5 — Baseline eval (untrained adapters)

Before any training, measure where we are. This gives us the "before" number for the comparison plot.

In [ ]:
# Cell 5 — baseline eval
def run_eval(eval_agents, label, n_per_repo=N_EVAL_EPISODES_PER_REPO):
    """Run eval episodes across all repos."""
    repos = ['maps_repo', 'payments_repo', 'comms_repo']
    all_results = []
    for repo_id in repos:
        repo_results = []
        for ep in range(n_per_repo):
            try:
                with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                    try:
                        obs = env.reset(repo_id=repo_id)
                    except TypeError:
                        obs = env.reset()
                    total_reward = 0.0
                    transcript = []
                    while not obs.done and obs.turn_index < 7:
                        agent = eval_agents[obs.current_role]
                        action = agent(obs)
                        transcript.append({
                            'turn': obs.turn_index,
                            'role': obs.current_role,
                            'action_type': action.action_type,
                        })
                        obs = env.step(action)
                        total_reward += float(obs.reward)
                    state = env.state()
                    tr = state.test_results or {}
                    repo_results.append({
                        'phase': label,
                        'repo': repo_id,
                        'episode': ep,
                        'total_reward': total_reward,
                        'tests_passed': sum(1 for v in tr.values() if v),
                        'tests_total': len(tr),
                        'success': bool(tr and all(tr.values())),
                        'turns': state.step_count,
                        'terminated': getattr(state, 'terminated_reason', None),
                        'transcript': transcript,
                    })
            except Exception as e:
                print(f'  {repo_id} ep {ep} crashed: {e}')

        if repo_results:
            pr = sum(r['success'] for r in repo_results) / len(repo_results)
            mr = sum(r['total_reward'] for r in repo_results) / len(repo_results)
            print(f'  {repo_id}: pass_rate={pr:.0%} mean_reward={mr:+.3f}')
        all_results.extend(repo_results)
    return all_results


print('=== Baseline eval (untrained adapters) ===')
baseline_results = run_eval(agents, 'baseline')
save_jsonl(LOGS / f'baseline_{RUN_ID}.jsonl', baseline_results)

if baseline_results:
    b_pass = sum(r['success'] for r in baseline_results) / len(baseline_results)
    b_reward = sum(r['total_reward'] for r in baseline_results) / len(baseline_results)
    print(f'\nBaseline overall: pass_rate={b_pass:.1%}, mean_reward={b_reward:+.3f}')
    print(f'  episodes: {len(baseline_results)}')

## Cell 6 — SFT Lead adapter on teacher demonstrations

This is the core training cell. We use TRL's SFTTrainer with the teacher demos.
Unlike GRPO, SFT directly imitates the expert. No reward variance issues.

In [ ]:
# Cell 6 — SFT the Lead adapter
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

# Build dataset in chat format that the tokenizer understands
def build_sft_rows(demos):
    rows = []
    for d in demos:
        # The model should learn: given prompt, produce the action JSON
        rows.append({
            'prompt': d['prompt'],
            'completion': d['completion'],
        })
    return rows

lead_rows = build_sft_rows(lead_demos)
print(f'Lead SFT rows: {len(lead_rows)}')

if not lead_rows:
    print('No lead demos - skipping training')
else:
    # Format as instruction-tuning text
    def format_for_sft(example):
        text = example['prompt'] + example['completion'] + tokenizer.eos_token
        return {'text': text}

    ds_lead = Dataset.from_list(lead_rows).map(format_for_sft)

    # === Set Lead as active and trainable, freeze Auditor ===
    model.set_adapter('lead_adapter')
    for n, p in model.named_parameters():
        if 'lead_adapter' in n:
            p.requires_grad = True
        else:
            p.requires_grad = False

    # === Backup callback ===
    class HFBackupCallback(TrainerCallback):
        def __init__(self, repo_id, save_every=10):
            self.repo_id = repo_id
            self.save_every = save_every
            self.last = -1
        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step > 0 and state.global_step % self.save_every == 0 and state.global_step != self.last:
                self.last = state.global_step
                snap = ARTIFACTS / f'lead_step_{state.global_step}'
                snap.mkdir(parents=True, exist_ok=True)
                try:
                    model.save_pretrained(str(snap))
                    push_to_hub_safe(snap, self.repo_id, f'step {state.global_step}', path_in_repo=f'step_{state.global_step}')
                    import shutil
                    shutil.rmtree(snap, ignore_errors=True)
                except Exception as e:
                    print(f'  backup at step {state.global_step} failed: {e}')

    if USE_WANDB:
        import wandb
        wandb.init(project=WANDB_PROJECT, name=f'sft-lead-{RUN_ID}', reinit=True)

    config = SFTConfig(
        output_dir=str(ARTIFACTS / 'sft_lead_ckpt'),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=3,  # 3 passes over the small expert set
        learning_rate=2e-4,  # standard for LoRA SFT
        logging_steps=2,
        save_steps=50,
        max_length=2048,
        report_to='wandb' if USE_WANDB else 'none',
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
    )

    print(f'Training Lead adapter on {len(ds_lead)} expert demos for 3 epochs...')
    trainer = SFTTrainer(
        model=model,
        train_dataset=ds_lead,
        args=config,
        callbacks=[HFBackupCallback(LEAD_REPO, save_every=10)],
    )
    trainer.train()

    # Final save
    final_dir = ARTIFACTS / 'lead_adapter_final'
    final_dir.mkdir(parents=True, exist_ok=True)
    model.set_adapter('lead_adapter')
    model.save_pretrained(str(final_dir))
    push_to_hub_safe(final_dir, LEAD_REPO, 'final lead adapter')

    if USE_WANDB:
        wandb.finish()
    print('OK: Lead adapter trained and pushed to Hub')

## Cell 7 — SFT Auditor adapter on teacher demonstrations

In [ ]:
# Cell 7 — SFT the Auditor adapter
auditor_rows = build_sft_rows(auditor_demos)
print(f'Auditor SFT rows: {len(auditor_rows)}')

if not auditor_rows:
    print('No auditor demos - skipping training')
else:
    ds_auditor = Dataset.from_list(auditor_rows).map(format_for_sft)

    # Switch to Auditor adapter, freeze Lead
    model.set_adapter('auditor_adapter')
    for n, p in model.named_parameters():
        if 'auditor_adapter' in n:
            p.requires_grad = True
        else:
            p.requires_grad = False

    if USE_WANDB:
        import wandb
        wandb.init(project=WANDB_PROJECT, name=f'sft-auditor-{RUN_ID}', reinit=True)

    config = SFTConfig(
        output_dir=str(ARTIFACTS / 'sft_auditor_ckpt'),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=2,
        save_steps=50,
        max_length=2048,
        report_to='wandb' if USE_WANDB else 'none',
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
    )

    print(f'Training Auditor adapter on {len(ds_auditor)} expert demos for 3 epochs...')
    trainer = SFTTrainer(
        model=model,
        train_dataset=ds_auditor,
        args=config,
        callbacks=[HFBackupCallback(AUDITOR_REPO, save_every=10)],
    )
    trainer.train()

    final_dir = ARTIFACTS / 'auditor_adapter_final'
    final_dir.mkdir(parents=True, exist_ok=True)
    model.set_adapter('auditor_adapter')
    model.save_pretrained(str(final_dir))
    push_to_hub_safe(final_dir, AUDITOR_REPO, 'final auditor adapter')

    if USE_WANDB:
        wandb.finish()
    print('OK: Auditor adapter trained and pushed to Hub')

## Cell 8 — Trained eval (the moment of truth)

In [ ]:
# Cell 8 — eval with trained adapters
print('=== Trained eval (post-SFT adapters) ===')

# The model already has the trained adapters active in memory
# but make sure agents are using them
trained_results = run_eval(agents, 'trained')
save_jsonl(LOGS / f'trained_{RUN_ID}.jsonl', trained_results)

if trained_results:
    t_pass = sum(r['success'] for r in trained_results) / len(trained_results)
    t_reward = sum(r['total_reward'] for r in trained_results) / len(trained_results)
    print(f'\n=== TRAINED RESULTS ===')
    print(f'Pass rate: {t_pass:.1%}')
    print(f'Mean reward: {t_reward:+.3f}')

    if 'b_pass' in dir():
        delta_pass = t_pass - b_pass
        delta_reward = t_reward - b_reward
        print(f'\n=== IMPROVEMENT vs BASELINE ===')
        print(f'Pass rate delta: {delta_pass:+.1%}')
        print(f'Mean reward delta: {delta_reward:+.3f}')
        print(f'\nBefore: {b_pass:.1%} pass, {b_reward:+.3f} reward')
        print(f'After:  {t_pass:.1%} pass, {t_reward:+.3f} reward')

## Cell 9 — Plots and final bundle

In [ ]:
# Cell 9 — judge-friendly plots + bundle
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import shutil

if baseline_results and trained_results:
    # Plot 1: pass rate comparison
    b_pr = sum(r['success'] for r in baseline_results) / len(baseline_results)
    t_pr = sum(r['success'] for r in trained_results) / len(trained_results)
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(['Baseline', 'Trained'], [b_pr, t_pr], color=['#999', '#1f77b4'])
    for bar, val in zip(bars, [b_pr, t_pr]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.0%}',
                ha='center', fontweight='bold', fontsize=12)
    ax.set_ylabel('Pass rate')
    ax.set_title(f'Pass rate (n={len(baseline_results)} baseline, {len(trained_results)} trained)')
    ax.set_ylim(0, max(1.0, max(b_pr, t_pr) + 0.1))
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS / 'pass_rate.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Plot 2: mean reward comparison
    b_r = sum(r['total_reward'] for r in baseline_results) / len(baseline_results)
    t_r = sum(r['total_reward'] for r in trained_results) / len(trained_results)
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(['Baseline', 'Trained'], [b_r, t_r], color=['#999', '#1f77b4'])
    for bar, val in zip(bars, [b_r, t_r]):
        offset = 0.1 if val >= 0 else -0.2
        ax.text(bar.get_x() + bar.get_width()/2, val + offset, f'{val:+.2f}',
                ha='center', fontweight='bold')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_ylabel('Mean episode reward')
    ax.set_title('Mean reward per episode')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS / 'mean_reward.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Plot 3: per-repo pass rate
    by_repo_b = defaultdict(list)
    by_repo_t = defaultdict(list)
    for r in baseline_results:
        by_repo_b[r['repo']].append(r['success'])
    for r in trained_results:
        by_repo_t[r['repo']].append(r['success'])
    repos = sorted(set(by_repo_b.keys()) | set(by_repo_t.keys()))
    if repos:
        x = np.arange(len(repos))
        b_rates = [np.mean(by_repo_b.get(r, [0])) for r in repos]
        t_rates = [np.mean(by_repo_t.get(r, [0])) for r in repos]
        fig, ax = plt.subplots(figsize=(9, 5))
        width = 0.35
        ax.bar(x - width/2, b_rates, width, label='Baseline', color='#999')
        ax.bar(x + width/2, t_rates, width, label='Trained', color='#1f77b4')
        ax.set_xticks(x)
        ax.set_xticklabels(repos)
        ax.set_ylabel('Pass rate')
        ax.set_title('Per-repo pass rate, baseline vs trained')
        ax.legend()
        ax.set_ylim(0, max(1.0, max(b_rates + t_rates) + 0.1))
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(PLOTS / 'per_repo.png', dpi=150, bbox_inches='tight')
        plt.close()

    # Plot 4: reward distribution
    b_rewards = [r['total_reward'] for r in baseline_results]
    t_rewards = [r['total_reward'] for r in trained_results]
    fig, ax = plt.subplots(figsize=(9, 5))
    bins = np.linspace(min(min(b_rewards), min(t_rewards)) - 0.1,
                       max(max(b_rewards), max(t_rewards)) + 0.1, 20)
    ax.hist(b_rewards, bins=bins, alpha=0.5, label='Baseline', color='#999')
    ax.hist(t_rewards, bins=bins, alpha=0.5, label='Trained', color='#1f77b4')
    ax.set_xlabel('Total episode reward')
    ax.set_ylabel('Episode count')
    ax.set_title('Reward distribution')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS / 'reward_dist.png', dpi=150, bbox_inches='tight')
    plt.close()

    print('OK: 4 plots saved to plots/')

    # Push everything to Hub
    push_to_hub_safe(PLOTS, ARTIFACTS_REPO, f'plots {RUN_ID}', path_in_repo=f'plots_{RUN_ID}')
    push_to_hub_safe(LOGS, ARTIFACTS_REPO, f'logs {RUN_ID}', path_in_repo=f'logs_{RUN_ID}')

# === Bundle and download ===
bundle_dir = BASE / f'sdk_sovereign_distilled_{RUN_ID}'
bundle_dir.mkdir(exist_ok=True)
for src in ['logs', 'artifacts', 'plots']:
    s = BASE / src
    if s.exists():
        shutil.copytree(s, bundle_dir / src, dirs_exist_ok=True)

tar_path = f'/content/sdk_sovereign_distilled_{RUN_ID}.tar.gz'
subprocess.run(['tar', '-czf', tar_path, '-C', str(BASE), bundle_dir.name], check=True)
print(f'\nBundle: {tar_path} ({os.path.getsize(tar_path)/1e6:.1f} MB)')

# === Final summary ===
print('\n' + '='*70)
print('PERMANENT STORAGE')
print('='*70)
print(f'\nLead adapter:    https://huggingface.co/{LEAD_REPO}')
print(f'Auditor adapter: https://huggingface.co/{AUDITOR_REPO}')
print(f'Artifacts:       https://huggingface.co/{ARTIFACTS_REPO}')
if USE_WANDB:
    print(f'W&B:             https://wandb.ai/<entity>/{WANDB_PROJECT}')

# Try to download
try:
    from google.colab import files
    files.download(tar_path)
except Exception as e:
    print(f'\nDirect download failed: {e}. Use HF Hub links above.')

print('\nDONE')